# CKD EDA — read-only profiling notebook

**Read-only contract (Pattern N1):** this notebook observes the raw CKD data through the existing pipeline functions only.
It imports `load_raw_data`, `summarize`, and `clean_raw` from `src.*` and never re-implements quirk handling
(`?` to NaN, whitespace stripping, dtype coercion). It never fills, imputes, encodes, or fits anything —
working copies are named `eda_df` / `plot_df` and are only filtered or coerced for plotting.
Every missingness number shown here comes from `summarize()`, the single source of truth.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # kernel cwd is notebooks/ -> repo root (Pattern N1 shim)

ROOT = Path("..").resolve() if Path("..", "src").exists() else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ingestion.load_data import load_raw_data, summarize
from src.preprocessing.preprocess import NUMERIC_COLS, CATEGORICAL_COLS, clean_raw

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
eda_df = load_raw_data()  # configured raw path; quirks already normalized upstream

summary = summarize(eda_df)  # single source of truth for missingness numbers
summary

2026-09-19 21:59:14.422 | INFO     | src.ingestion.load_data:load_raw_data:50 - Loaded raw data: 400 rows x 26 cols from C:\Users\Lalit\OneDrive\Desktop\TY\EDI\Project\Corvus\Corvus\data\raw\kidney_disease.csv


,dtype,missing_count,missing_pct,n_unique
rbc,str,152,38.00,2
rc,str,131,32.75,48
wc,str,106,26.50,89
pot,float64,88,22.00,40
sod,float64,87,21.75,34
pcv,str,71,17.75,42
pc,str,65,16.25,2
hemo,float64,52,13.00,115
su,float64,49,12.25,6
sg,float64,47,11.75,5


In [3]:
missing_pct = eda_df.isna().mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
missing_pct.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("missing %")
ax.set_title("Per-column missingness (raw CKD data)")
fig.savefig(FIG_DIR / "missingness_bar.png", bbox_inches="tight")
plt.close(fig)
missing_pct

rbc               38.00
rc                32.75
wc                26.50
pot               22.00
sod               21.75
pcv               17.75
pc                16.25
hemo              13.00
su                12.25
sg                11.75
al                11.50
bgr               11.00
bu                 4.75
sc                 4.25
bp                 3.00
age                2.25
pcc                1.00
ba                 1.00
dm                 0.50
htn                0.50
cad                0.50
appet              0.25
ane                0.25
pe                 0.25
id                 0.00
classification     0.00
dtype: float64

In [4]:
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(eda_df.isna(), cbar_kws={"label": "missing"}, yticklabels=False, ax=ax)
ax.set_title("Missing-value map (yellow = missing)")
fig.savefig(FIG_DIR / "missingness_heatmap.png", bbox_inches="tight")
plt.close(fig)

## Target-aware distributions

`plot_df` reuses the pipeline's own `clean_raw` cleaning (target labels mapped to 0/1 by the pipeline,
no manual recoding here). Hemoglobin shows class separation; red-blood-cell counts are compared per class.

In [5]:
plot_df = clean_raw(load_raw_data())  # reuse pipeline cleaning — never recode labels by hand

grid = sns.displot(plot_df, x="hemo", hue="classification", kind="kde", fill=True)
grid.fig.suptitle("Hemoglobin distribution by CKD class", y=1.02)
grid.fig.savefig(FIG_DIR / "hemo_kde_by_class.png", bbox_inches="tight")
plt.close("all")

2026-09-19 21:59:15.072 | INFO     | src.ingestion.load_data:load_raw_data:50 - Loaded raw data: 400 rows x 26 cols from C:\Users\Lalit\OneDrive\Desktop\TY\EDI\Project\Corvus\Corvus\data\raw\kidney_disease.csv


In [6]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=plot_df, x="rbc", hue="classification", ax=ax)
ax.set_title("RBC counts by CKD class")
fig.savefig(FIG_DIR / "rbc_count_by_class.png", bbox_inches="tight")
plt.close(fig)

## Numeric correlations

Numeric columns coerced with `pd.to_numeric(errors="coerce")` (mirroring `clean_raw`), Pearson matrix
on a fixed [-1, 1] diverging scale with the upper triangle masked.

In [7]:
num = eda_df[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce")
corr = num.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag",
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Numeric feature correlations (Pearson)")
fig.savefig(FIG_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.close(fig)
corr

,age,bp,sg,al,su,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc
age,1.000000,0.159480,-0.191096,0.122091,0.220866,0.244992,0.196985,0.132531,-0.100046,0.058377,-0.192928,-0.242119,0.118339,-0.268896
bp,0.159480,1.000000,-0.218836,0.160689,0.222576,0.160193,0.188517,0.146222,-0.116422,0.075151,-0.306540,-0.326319,0.029753,-0.261936
sg,-0.191096,-0.218836,1.000000,-0.469760,-0.296234,-0.374710,-0.314295,-0.361473,0.412190,-0.072787,0.602582,0.603560,-0.236215,0.579476
al,0.122091,0.160689,-0.469760,1.000000,0.269305,0.379464,0.453528,0.399198,-0.459896,0.129038,-0.634632,-0.611891,0.231989,-0.566437
su,0.220866,0.222576,-0.296234,0.269305,1.000000,0.717827,0.168583,0.223244,-0.131776,0.219450,-0.224775,-0.239189,0.184893,-0.237448
bgr,0.244992,0.160193,-0.374710,0.379464,0.717827,1.000000,0.143322,0.114875,-0.267848,0.066966,-0.306189,-0.301385,0.150015,-0.281541
bu,0.196985,0.188517,-0.314295,0.453528,0.168583,0.143322,1.000000,0.586368,-0.323054,0.357049,-0.610360,-0.607621,0.050462,-0.579087
sc,0.132531,0.146222,-0.361473,0.399198,0.223244,0.114875,0.586368,1.000000,-0.690158,0.326107,-0.401670,-0.404193,-0.006390,-0.400852
sod,-0.100046,-0.116422,0.412190,-0.459896,-0.131776,-0.267848,-0.323054,-0.690158,1.000000,0.097887,0.365183,0.376914,0.007277,0.344873
pot,0.058377,0.075151,-0.072787,0.129038,0.219450,0.066966,0.357049,0.326107,0.097887,1.000000,-0.133746,-0.163182,-0.105576,-0.158309


## Takeaways

1. Highest missingness is in `rbc` (38.00%), followed by `rc` (32.75%) and `wc` (26.50%) —
   all from the red/white blood-cell block, which motivates the pipeline's median/most-frequent imputation.
2. Strongest numeric correlation is hemoglobin–packed-cell-volume (Pearson r ~ 0.90); the anemia block
   (hemo, pcv, rc) is highly intercorrelated, so the model will see redundant red-cell signals.